# Week 8 Jupyter Notebook: K-Nearest Neighbors and Distance Metrics
## Digital Customer Journey and E-Commerce Retention Analytics

**Student:** Javed Aamer

### Goal of this notebook

This week I use K-Nearest Neighbors (KNN) to study the same customer-level e-commerce data used in the earlier capstone work.

I use KNN in two ways:

- **classification:** predict whether a customer churns;
- **regression:** predict `OrderCount` by averaging nearby customers.

The main focus is not simply whether KNN works. I want to understand how a distance-based model changes when I alter **feature scaling**, **the number of neighbors (`k`)**, and the **definition of distance**.

The distance comparisons include Euclidean, Manhattan, Minkowski, and cosine distance. I also examine why one large outlier dimension affects these metrics differently.

## Concepts covered

This notebook directly applies the Week 8 ideas:

- KNN classification by majority vote;
- KNN regression by neighbor averaging;
- Euclidean distance;
- Manhattan distance;
- Minkowski distance with different values of `p`;
- cosine distance;
- feature scaling for distance-based models;
- sensitivity to an outlier dimension;
- tuning `k`;
- the bias-variance tradeoff as `k` changes.

I keep the analysis tied to the capstone rather than using a separate toy dataset.

## 1. Questions I wanted to answer

1. How much does feature scaling change KNN performance on this customer data?
2. What happens when `k` is very small versus larger?
3. Which distance metric works best for churn classification?
4. Do Euclidean and Manhattan distance react differently to a single large feature difference?
5. Does cosine distance behave differently because it focuses on direction rather than vector magnitude?
6. Are the best KNN choices for churn classification also the best choices for `OrderCount` regression?
7. Is KNN competitive enough to be useful for this project, or is its main value diagnostic?

In [1]:
# Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    roc_curve,
)
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import euclidean, cityblock, minkowski

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
# Load the cleaned customer-level dataset
candidate_paths = [Path("data/churn_clean.csv"), Path("churn_clean.csv")]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find churn_clean.csv. Keep the data folder next to this notebook."
    )

churn = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {churn.shape[0]:,} rows x {churn.shape[1]} columns")
print(f"Unique CustomerID values: {churn['CustomerID'].nunique():,}")
print(f"Duplicate CustomerID rows: {churn['CustomerID'].duplicated().sum():,}")
display(churn.head())

Loaded: data/churn_clean.csv
Shape: 5,630 rows x 20 columns
Unique CustomerID values: 5,630
Duplicate CustomerID rows: 0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0000,Mobile Phone,3,6.0000,Debit Card,Female,3.0000,3,Laptop & Accessory,2,Single,9,1,11.0000,1.0000,1.0000,5.0000,159.9300
1,50002,1,9.0000,Phone,1,8.0000,UPI,Male,3.0000,4,Mobile,3,Single,7,1,15.0000,0.0000,1.0000,0.0000,120.9000
2,50003,1,9.0000,Phone,1,30.0000,Debit Card,Male,2.0000,4,Mobile,3,Single,6,1,14.0000,0.0000,1.0000,3.0000,120.2800
3,50004,1,0.0000,Phone,3,15.0000,Debit Card,Male,2.0000,4,Laptop & Accessory,5,Single,8,0,23.0000,0.0000,1.0000,3.0000,134.0700
4,50005,1,0.0000,Phone,1,12.0000,CC,Male,3.0000,3,Mobile,5,Single,3,0,11.0000,1.0000,1.0000,3.0000,129.6000


## 2. Modeling setup

For the churn classifier, I use the same operational feature set as in the earlier classification work.

### Classification response
- `Churn` (`0` = stayed, `1` = churned)

### Numeric predictors
`Tenure`, `CityTier`, `WarehouseToHome`, `HourSpendOnApp`, `NumberOfDeviceRegistered`, `SatisfactionScore`, `NumberOfAddress`, `Complain`, `OrderAmountHikeFromlastYear`, `CouponUsed`, `OrderCount`, `DaySinceLastOrder`, and `CashbackAmount`.

### Categorical predictors
`PreferredLoginDevice`, `PreferredPaymentMode`, and `PreferedOrderCat`.

`CustomerID` is excluded because it is an identifier. I also leave `Gender` and `MaritalStatus` out of the operational feature set.

For the regression analysis, `OrderCount` becomes the response, so it is removed from the predictor list.

In [3]:
class_target = "Churn"
reg_target = "OrderCount"

numeric_features = [
    "Tenure", "CityTier", "WarehouseToHome", "HourSpendOnApp",
    "NumberOfDeviceRegistered", "SatisfactionScore", "NumberOfAddress",
    "Complain", "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount",
]

categorical_features = [
    "PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat",
]

classification_features = numeric_features + categorical_features
reg_numeric_features = [x for x in numeric_features if x != reg_target]
regression_features = reg_numeric_features + categorical_features

audit = pd.DataFrame({
    "dtype": churn[classification_features + [class_target]].dtypes.astype(str),
    "missing": churn[classification_features + [class_target]].isna().sum(),
    "unique": churn[classification_features + [class_target]].nunique(dropna=False),
})
display(audit)

,dtype,missing,unique
Tenure,float64,0,36
CityTier,int64,0,3
WarehouseToHome,float64,0,34
HourSpendOnApp,float64,0,6
NumberOfDeviceRegistered,int64,0,6
SatisfactionScore,int64,0,5
NumberOfAddress,int64,0,15
Complain,int64,0,2
OrderAmountHikeFromlastYear,float64,0,16
CouponUsed,float64,0,17


## 3. Why scaling matters for KNN

KNN decides which observations are "near" each other. If one numeric feature naturally has much larger values than another, that feature can dominate the distance calculation even when it is not more important to the prediction.

I first look at the original standard deviations. The wide spread is a warning that raw Euclidean or Manhattan distance will not treat the numeric dimensions comparably.

In [4]:
raw_scale = churn[numeric_features].std().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(raw_scale.index, raw_scale.values)
ax.set_xscale("log")
ax.set_xlabel("Standard deviation before scaling (log scale)")
ax.set_ylabel("Numeric predictor")
ax.set_title("KNN starts with numeric features on very different scales")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

display(churn[numeric_features].agg(["min", "max", "mean", "std"]).T)

<Figure size 800x600 with 1 Axes>

,min,max,mean,std
Tenure,0.0000,61.0000,10.1341,8.3580
CityTier,1.0000,3.0000,1.6547,0.9154
WarehouseToHome,5.0000,127.0000,15.5668,8.3460
HourSpendOnApp,0.0000,5.0000,2.9346,0.7055
NumberOfDeviceRegistered,1.0000,6.0000,3.6890,1.0240
SatisfactionScore,1.0000,5.0000,3.0668,1.3802
NumberOfAddress,1.0000,22.0000,4.2140,2.5836
Complain,0.0000,1.0000,0.2849,0.4514
OrderAmountHikeFromlastYear,11.0000,26.0000,15.6746,3.5911
CouponUsed,0.0000,16.0000,1.7169,1.8576


In [5]:
def make_preprocessor(numeric_cols, scale=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_features),
    ])


def classification_metrics(y_true, probability, prediction):
    return {
        "ROC-AUC": roc_auc_score(y_true, probability),
        "PR-AUC": average_precision_score(y_true, probability),
        "Accuracy": accuracy_score(y_true, prediction),
        "Balanced accuracy": balanced_accuracy_score(y_true, prediction),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
    }


def regression_metrics(y_true, prediction):
    return {
        "R2": r2_score(y_true, prediction),
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": mean_squared_error(y_true, prediction) ** 0.5,
    }


def make_knn_classifier(k, metric_name, p=None):
    kwargs = {"n_neighbors": k, "weights": "uniform"}
    if metric_name == "cosine":
        kwargs.update(metric="cosine", algorithm="brute")
    elif metric_name == "minkowski":
        kwargs.update(metric="minkowski", p=p)
    else:
        kwargs.update(metric=metric_name)
    return KNeighborsClassifier(**kwargs)


def make_knn_regressor(k, metric_name, p=None):
    kwargs = {"n_neighbors": k, "weights": "uniform"}
    if metric_name == "cosine":
        kwargs.update(metric="cosine", algorithm="brute")
    elif metric_name == "minkowski":
        kwargs.update(metric="minkowski", p=p)
    else:
        kwargs.update(metric=metric_name)
    return KNeighborsRegressor(**kwargs)

# Part A: KNN Classification for Churn

## 4. Stratified train, validation, and test split

I use a 60/20/20 stratified split so the churn rate remains similar across the three partitions.

- training data fit the preprocessing and KNN model;
- validation data select `k` and the distance metric;
- the test set is held back until those choices are fixed.

In [6]:
X_cls = churn[classification_features].copy()
y_cls = churn[class_target].astype(int).copy()

X_dev_cls, X_test_cls, y_dev_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cls
)
X_train_cls, X_val_cls, y_train_cls, y_val_cls = train_test_split(
    X_dev_cls, y_dev_cls, test_size=0.25, random_state=RANDOM_STATE, stratify=y_dev_cls
)

split_summary = pd.DataFrame({
    "Rows": [len(X_train_cls), len(X_val_cls), len(X_test_cls)],
    "Churn rate": [y_train_cls.mean(), y_val_cls.mean(), y_test_cls.mean()],
}, index=["Training", "Validation", "Test"])
display(split_summary)

,Rows,Churn rate
Training,3378,0.1684
Validation,1126,0.1679
Test,1126,0.1687


## 5. Scaled versus unscaled KNN

To isolate the effect of scaling, I first keep the model fixed at `k = 11` with Euclidean distance and change only the preprocessing.

If scaling matters, the standardized version should identify more meaningful neighbors because a large-unit feature such as `CashbackAmount` no longer dominates a smaller-unit feature simply because of its units.

In [7]:
scaling_rows = []
scaling_models = {}

for label, scale in [("Unscaled", False), ("Standardized", True)]:
    pre = make_preprocessor(numeric_features, scale=scale)
    Xtr = pre.fit_transform(X_train_cls)
    Xv = pre.transform(X_val_cls)

    model = make_knn_classifier(11, "euclidean")
    model.fit(Xtr, y_train_cls)
    prob = model.predict_proba(Xv)[:, 1]
    pred = model.predict(Xv)

    scaling_rows.append({"Preprocessing": label, **classification_metrics(y_val_cls, prob, pred)})
    scaling_models[label] = (pre, model)

scaling_results = pd.DataFrame(scaling_rows)
display(scaling_results)

,Preprocessing,ROC-AUC,PR-AUC,Accuracy,Balanced accuracy,Precision,Recall,F1
0,Unscaled,0.8213,0.4790,0.8508,0.6316,0.6129,0.3016,0.4043
1,Standardized,0.9003,0.6682,0.8792,0.6719,0.8193,0.3598,0.5000


In [8]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = scaling_results.set_index("Preprocessing")[["ROC-AUC", "PR-AUC", "F1"]]
x = np.arange(len(plot_df.columns))
width = 0.35
ax.bar(x - width/2, plot_df.loc["Unscaled"], width, label="Unscaled")
ax.bar(x + width/2, plot_df.loc["Standardized"], width, label="Standardized")
ax.set_xticks(x)
ax.set_xticklabels(plot_df.columns)
ax.set_ylabel("Validation score")
ax.set_title("Feature scaling changes which customers KNN considers nearby")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

## 6. Tune `k`: bias versus variance

A very small `k` makes KNN highly local. That can fit small details of the training data, which means lower bias but higher variance.

A larger `k` averages over more neighbors. The boundary becomes smoother, which usually lowers variance but can introduce bias if dissimilar customers are averaged together.

I standardize the numeric predictors and compare training and validation ROC-AUC across odd values of `k`.

In [9]:
scaled_pre_cls = make_preprocessor(numeric_features, scale=True)
X_train_cls_s = scaled_pre_cls.fit_transform(X_train_cls)
X_val_cls_s = scaled_pre_cls.transform(X_val_cls)

k_values = [1, 3, 5, 7, 11, 15, 21, 31]
k_rows = []

for k in k_values:
    model = make_knn_classifier(k, "euclidean")
    model.fit(X_train_cls_s, y_train_cls)

    train_prob = model.predict_proba(X_train_cls_s)[:, 1]
    val_prob = model.predict_proba(X_val_cls_s)[:, 1]

    k_rows.append({
        "k": k,
        "Training ROC-AUC": roc_auc_score(y_train_cls, train_prob),
        "Validation ROC-AUC": roc_auc_score(y_val_cls, val_prob),
    })

k_results = pd.DataFrame(k_rows)
best_k_euclidean = int(k_results.loc[k_results["Validation ROC-AUC"].idxmax(), "k"])
print(f"Best Euclidean k on validation ROC-AUC: {best_k_euclidean}")
display(k_results)

Best Euclidean k on validation ROC-AUC: 21


,k,Training ROC-AUC,Validation ROC-AUC
0,1,1.0000,0.8829
1,3,0.9908,0.8919
2,5,0.9769,0.9007
3,7,0.9636,0.9013
4,11,0.9453,0.9003
5,15,0.9317,0.9011
6,21,0.9176,0.9014
7,31,0.9015,0.8957


In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_results["k"], k_results["Training ROC-AUC"], marker="o", label="Training")
ax.plot(k_results["k"], k_results["Validation ROC-AUC"], marker="o", label="Validation")
ax.axvline(best_k_euclidean, linestyle="--", label=f"Best validation k = {best_k_euclidean}")
ax.set_xlabel("Number of neighbors, k")
ax.set_ylabel("ROC-AUC")
ax.set_title("Small k fits the training data more aggressively")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

## 7. Compare distance metrics

I keep feature scaling fixed and tune `k` separately for each distance definition.

- **Euclidean** squares coordinate differences before combining them, so one large difference can matter strongly.
- **Manhattan** adds absolute differences and is less sensitive to one extreme coordinate.
- **Minkowski** generalizes both: `p = 1` is Manhattan and `p = 2` is Euclidean. I test intermediate and larger values.
- **Cosine** focuses on vector direction rather than magnitude.

To make the comparison fair, each metric gets the same candidate values of `k`, and validation ROC-AUC chooses its best setting.

In [11]:
metric_specs = [
    ("Euclidean", "euclidean", None),
    ("Manhattan", "manhattan", None),
    ("Minkowski p=1.5", "minkowski", 1.5),
    ("Minkowski p=3", "minkowski", 3),
    ("Cosine", "cosine", None),
]

metric_tuning_rows = []
metric_paths = []

for label, metric_name, p in metric_specs:
    best = None
    for k in k_values:
        model = make_knn_classifier(k, metric_name, p)
        model.fit(X_train_cls_s, y_train_cls)
        prob = model.predict_proba(X_val_cls_s)[:, 1]
        auc = roc_auc_score(y_val_cls, prob)
        metric_paths.append({"Metric": label, "k": k, "Validation ROC-AUC": auc})
        if best is None or auc > best["Validation ROC-AUC"]:
            best = {"Metric": label, "k": k, "Validation ROC-AUC": auc, "metric_name": metric_name, "p": p}
    metric_tuning_rows.append(best)

metric_tuning = pd.DataFrame(metric_tuning_rows)
display(metric_tuning[["Metric", "k", "Validation ROC-AUC"]].sort_values("Validation ROC-AUC", ascending=False))

,Metric,k,Validation ROC-AUC
4,Cosine,7,0.9093
2,Minkowski p=1.5,7,0.9054
1,Manhattan,15,0.9031
0,Euclidean,21,0.9014
3,Minkowski p=3,5,0.8983


In [12]:
metric_paths_df = pd.DataFrame(metric_paths)
fig, ax = plt.subplots(figsize=(9, 6))
for metric in metric_paths_df["Metric"].unique():
    d = metric_paths_df[metric_paths_df["Metric"] == metric]
    ax.plot(d["k"], d["Validation ROC-AUC"], marker="o", label=metric)
ax.set_xlabel("Number of neighbors, k")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("The best k depends on how distance is defined")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 900x600 with 1 Axes>

## 8. Why an outlier dimension changes the answer

The lecture notes emphasize that Euclidean and Manhattan distance respond differently when one candidate neighbor is very far away on a single feature.

I illustrate that in standardized units. Starting from the same reference vector:

- Candidate A differs by **8 standard deviations in one dimension**.
- Candidate B differs by **3 standard deviations in three dimensions**.

This is not another model fit. It is a diagnostic to show what each distance definition means geometrically.

In [13]:
reference = np.array([1.0, 1.0, 1.0])
candidate_single_outlier = np.array([9.0, 1.0, 1.0])   # difference: [8, 0, 0]
candidate_balanced = np.array([4.0, 4.0, 4.0])          # difference: [3, 3, 3]

rows = []
for name, metric_fn in [
    ("Euclidean", lambda a,b: euclidean(a,b)),
    ("Manhattan", lambda a,b: cityblock(a,b)),
    ("Minkowski p=1.5", lambda a,b: minkowski(a,b,1.5)),
    ("Minkowski p=3", lambda a,b: minkowski(a,b,3)),
    ("Cosine", lambda a,b: float(cosine_distances(a.reshape(1,-1), b.reshape(1,-1))[0,0])),
]:
    da = metric_fn(reference, candidate_single_outlier)
    db = metric_fn(reference, candidate_balanced)
    rows.append({
        "Metric": name,
        "Single large dimension": da,
        "Several moderate dimensions": db,
        "Closer candidate": "Single-outlier A" if da < db else "Balanced B",
    })

outlier_distance_results = pd.DataFrame(rows)
display(outlier_distance_results)

,Metric,Single large dimension,Several moderate dimensions,Closer candidate
0,Euclidean,8.0000,5.1962,Balanced B
1,Manhattan,8.0000,9.0000,Single-outlier A
2,Minkowski p=1.5,8.0000,6.2403,Balanced B
3,Minkowski p=3,8.0000,4.3267,Balanced B
4,Cosine,0.3029,0.0000,Balanced B


In [14]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = outlier_distance_results.set_index("Metric")[["Single large dimension", "Several moderate dimensions"]]
x = np.arange(len(plot_df))
width = 0.36
ax.bar(x - width/2, plot_df["Single large dimension"], width, label="8 SD in one dimension")
ax.bar(x + width/2, plot_df["Several moderate dimensions"], width, label="3 SD in three dimensions")
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=15)
ax.set_ylabel("Distance")
ax.set_title("Distance metrics disagree about a single large feature difference")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 900x500 with 1 Axes>

## 9. Final classification comparison on untouched test data

The metric-specific `k` values are now fixed using the validation set.

I refit preprocessing on training plus validation data, fit each selected KNN model, and evaluate each one once on the untouched test set.

In [15]:
final_pre_cls = make_preprocessor(numeric_features, scale=True)
X_dev_cls_s = final_pre_cls.fit_transform(X_dev_cls)
X_test_cls_s = final_pre_cls.transform(X_test_cls)

final_cls_rows = []
final_cls_probabilities = {}
final_cls_predictions = {}

for _, row in metric_tuning.iterrows():
    label = row["Metric"]
    k = int(row["k"])
    metric_name = row["metric_name"]
    p = row["p"] if pd.notna(row["p"]) else None

    model = make_knn_classifier(k, metric_name, p)
    model.fit(X_dev_cls_s, y_dev_cls)
    prob = model.predict_proba(X_test_cls_s)[:, 1]
    pred = model.predict(X_test_cls_s)

    final_cls_probabilities[label] = prob
    final_cls_predictions[label] = pred
    final_cls_rows.append({"Metric": label, "k": k, **classification_metrics(y_test_cls, prob, pred)})

final_cls_results = pd.DataFrame(final_cls_rows).sort_values("ROC-AUC", ascending=False)
display(final_cls_results)

,Metric,k,ROC-AUC,PR-AUC,Accuracy,Balanced accuracy,Precision,Recall,F1
3,Minkowski p=3,5,0.9217,0.7142,0.9059,0.7651,0.8333,0.5526,0.6646
4,Cosine,7,0.9177,0.6996,0.8961,0.7445,0.7967,0.5158,0.6262
2,Minkowski p=1.5,7,0.9113,0.6960,0.8961,0.7278,0.8411,0.4737,0.6061
1,Manhattan,15,0.8947,0.6382,0.8757,0.6630,0.8125,0.3421,0.4815
0,Euclidean,21,0.8881,0.6477,0.8703,0.6473,0.7973,0.3105,0.4470


### Classification result

The final test results changed the ranking slightly from the validation set. **Cosine distance had the highest validation ROC-AUC (0.9093), but Minkowski distance with `p = 3` produced the strongest test result**, with ROC-AUC **0.9217**, PR-AUC **0.7142**, and F1 **0.6646** at `k = 5`.

That difference is useful rather than inconvenient: it shows that choosing among several similar distance rules from one validation split can be unstable. I would therefore treat Minkowski `p = 3` as the best result in this split, not as proof that it is universally the best metric.

Euclidean distance was weaker on the final test set (ROC-AUC 0.8881), even after scaling and tuning `k`. This suggests that the geometry of customer similarity matters, not only the number of neighbors.

In [16]:
fig, ax = plt.subplots(figsize=(7, 7))
for label, prob in final_cls_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test_cls, prob)
    auc_value = roc_auc_score(y_test_cls, prob)
    ax.plot(fpr, tpr, label=f"{label} ({auc_value:.3f})")
ax.plot([0,1], [0,1], linestyle="--", label="Random ranking")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("KNN churn classification: test ROC curves")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 700x700 with 1 Axes>

In [17]:
best_cls_metric = final_cls_results.iloc[0]["Metric"]
best_cls_k = int(final_cls_results.iloc[0]["k"])
best_cls_pred = final_cls_predictions[best_cls_metric]
cm = confusion_matrix(y_test_cls, best_cls_pred)

fig, ax = plt.subplots(figsize=(6,5))
ax.imshow(cm)
ax.set_xticks([0,1]); ax.set_xticklabels(["Predicted stay", "Predicted churn"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Actual stay", "Actual churn"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=12)
ax.set_title(f"Best KNN classifier: {best_cls_metric}, k={best_cls_k}")
plt.tight_layout()
plt.show()

<Figure size 600x500 with 1 Axes>

# Part B: KNN Regression for OrderCount

## 10. KNN regression

KNN regression uses the same idea of local similarity, but instead of taking a majority vote, it averages the target values of the nearest neighbors.

Here the target is `OrderCount`. I remove `OrderCount` from the predictors and ask whether customers with similar profiles also have similar ordering activity.

In [18]:
X_reg = churn[regression_features].copy()
y_reg = churn[reg_target].astype(float).copy()

X_dev_reg, X_test_reg, y_dev_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_dev_reg, y_dev_reg, test_size=0.25, random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train_reg):,}")
print(f"Validation rows: {len(X_val_reg):,}")
print(f"Test rows: {len(X_test_reg):,}")
display(y_reg.describe().to_frame().T)

Training rows: 3,378
Validation rows: 1,126
Test rows: 1,126


,count,mean,std,min,25%,50%,75%,max
OrderCount,"5,630.0000",2.9618,2.8792,1.0000,1.0000,2.0000,3.0000,16.0000


## 11. Scaling check for KNN regression

I repeat the same controlled comparison: Euclidean KNN with `k = 11`, once on unscaled numeric predictors and once after standardization.

In [19]:
reg_scaling_rows = []
for label, scale in [("Unscaled", False), ("Standardized", True)]:
    pre = make_preprocessor(reg_numeric_features, scale=scale)
    Xtr = pre.fit_transform(X_train_reg)
    Xv = pre.transform(X_val_reg)
    model = make_knn_regressor(11, "euclidean")
    model.fit(Xtr, y_train_reg)
    pred = model.predict(Xv)
    reg_scaling_rows.append({"Preprocessing": label, **regression_metrics(y_val_reg, pred)})

reg_scaling_results = pd.DataFrame(reg_scaling_rows)
display(reg_scaling_results)

,Preprocessing,R2,MAE,RMSE
0,Unscaled,0.3502,1.4699,2.4182
1,Standardized,0.4939,1.2357,2.1342


In [20]:
fig, ax = plt.subplots(figsize=(7,5))
ax.bar(reg_scaling_results["Preprocessing"], reg_scaling_results["RMSE"])
ax.set_ylabel("Validation RMSE")
ax.set_title("Scaling effect on KNN regression")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 700x500 with 1 Axes>

## 12. Tune k and distance for regression

For regression, lower validation RMSE is better. I standardize the numeric predictors and tune the same distance metrics over the same odd values of `k` used in classification.

In [21]:
scaled_pre_reg = make_preprocessor(reg_numeric_features, scale=True)
X_train_reg_s = scaled_pre_reg.fit_transform(X_train_reg)
X_val_reg_s = scaled_pre_reg.transform(X_val_reg)

reg_metric_rows = []
reg_metric_paths = []

for label, metric_name, p in metric_specs:
    best = None
    for k in k_values:
        model = make_knn_regressor(k, metric_name, p)
        model.fit(X_train_reg_s, y_train_reg)
        pred = model.predict(X_val_reg_s)
        rmse = mean_squared_error(y_val_reg, pred) ** 0.5
        reg_metric_paths.append({"Metric": label, "k": k, "Validation RMSE": rmse})
        if best is None or rmse < best["Validation RMSE"]:
            best = {"Metric": label, "k": k, "Validation RMSE": rmse, "metric_name": metric_name, "p": p}
    reg_metric_rows.append(best)

reg_metric_tuning = pd.DataFrame(reg_metric_rows)
display(reg_metric_tuning[["Metric", "k", "Validation RMSE"]].sort_values("Validation RMSE"))

,Metric,k,Validation RMSE
1,Manhattan,1,1.7656
2,Minkowski p=1.5,1,1.8247
0,Euclidean,1,1.8380
4,Cosine,1,1.8699
3,Minkowski p=3,1,1.9270


In [22]:
reg_paths_df = pd.DataFrame(reg_metric_paths)
fig, ax = plt.subplots(figsize=(9,6))
for metric in reg_paths_df["Metric"].unique():
    d = reg_paths_df[reg_paths_df["Metric"] == metric]
    ax.plot(d["k"], d["Validation RMSE"], marker="o", label=metric)
ax.set_xlabel("Number of neighbors, k")
ax.set_ylabel("Validation RMSE")
ax.set_title("KNN regression: k and distance both change prediction error")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 900x600 with 1 Axes>

## 13. Final regression comparison

After selecting each metric's `k` on validation RMSE, I refit on the combined training and validation data and evaluate once on the regression test set.

In [23]:
final_pre_reg = make_preprocessor(reg_numeric_features, scale=True)
X_dev_reg_s = final_pre_reg.fit_transform(X_dev_reg)
X_test_reg_s = final_pre_reg.transform(X_test_reg)

final_reg_rows = []
final_reg_predictions = {}

for _, row in reg_metric_tuning.iterrows():
    label = row["Metric"]
    k = int(row["k"])
    metric_name = row["metric_name"]
    p = row["p"] if pd.notna(row["p"]) else None
    model = make_knn_regressor(k, metric_name, p)
    model.fit(X_dev_reg_s, y_dev_reg)
    pred = model.predict(X_test_reg_s)
    final_reg_predictions[label] = pred
    final_reg_rows.append({"Metric": label, "k": k, **regression_metrics(y_test_reg, pred)})

final_reg_results = pd.DataFrame(final_reg_rows).sort_values("RMSE")
display(final_reg_results)

,Metric,k,R2,MAE,RMSE
1,Manhattan,1,0.7593,0.3597,1.4006
2,Minkowski p=1.5,1,0.7266,0.4094,1.4927
4,Cosine,1,0.7175,0.4307,1.5175
0,Euclidean,1,0.7069,0.4369,1.5456
3,Minkowski p=3,1,0.6312,0.5320,1.7338


### Regression result

The `OrderCount` regression produced a different answer. **Manhattan distance with `k = 1` was strongest**, reaching test RMSE **1.4006** and test R-squared **0.7593**. Every distance metric selected `k = 1` on the validation set.

That result suggests very local customer profiles contain useful information about ordering activity, but `k = 1` is also the highest-variance KNN setting because each prediction depends on a single training observation. I therefore view the result as promising but would confirm the choice with repeated cross-validation before using it operationally.

In [24]:
best_reg_metric = final_reg_results.iloc[0]["Metric"]
best_reg_k = int(final_reg_results.iloc[0]["k"])
best_reg_pred = final_reg_predictions[best_reg_metric]

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y_test_reg, best_reg_pred, alpha=0.35)
limits = [min(y_test_reg.min(), best_reg_pred.min()), max(y_test_reg.max(), best_reg_pred.max())]
ax.plot(limits, limits, linestyle="--", linewidth=1.5)
ax.set_xlim(limits); ax.set_ylim(limits)
ax.set_xlabel("Actual OrderCount")
ax.set_ylabel("Predicted OrderCount")
ax.set_title(f"Best KNN regression: {best_reg_metric}, k={best_reg_k}")
ax.grid(alpha=0.20)
plt.tight_layout()
plt.show()

<Figure size 600x600 with 1 Axes>

## 14. What I learned

### Scaling was essential

For churn classification with Euclidean KNN at `k = 11`, standardization increased validation ROC-AUC from **0.8213 to 0.9003** and PR-AUC from **0.4790 to 0.6682**. For `OrderCount` regression, validation RMSE improved from **2.4182 to 2.1342**. In this dataset, raw feature units clearly distorted neighborhood relationships.

### `k` exposed the bias-variance tradeoff

For Euclidean churn classification, `k = 1` achieved training ROC-AUC **1.0000** but validation ROC-AUC only **0.8829**. Validation performance improved as the neighborhood widened and peaked at `k = 21` with ROC-AUC **0.9014**, then declined by `k = 31`. This is a direct example of a very local model fitting the training data more aggressively than it generalizes.

### Distance choice changed which customers were considered similar

The standardized diagnostic illustrates the lecture concept clearly. Euclidean distance preferred a candidate with several moderate differences over one with an 8-standard-deviation difference in a single dimension. Manhattan made the opposite choice because it did not square the large coordinate difference. Minkowski shifted behavior as `p` changed, while cosine strongly favored the candidate pointing in the same direction.

The project results also showed that no one distance definition won every task. **Minkowski `p = 3` was best for churn classification on the final test set**, while **Manhattan was best for `OrderCount` regression**.

### Classification and regression define different notions of useful similarity

The best churn classifier used `k = 5` with Minkowski `p = 3`, whereas the best `OrderCount` regressor used Manhattan distance with `k = 1`. That difference makes sense because one task asks whether nearby customers share a class label, while the other asks whether nearby customers have similar numeric ordering activity.

### Project conclusion

KNN is useful for this capstone because it provides a transparent way to test whether customers with similar profiles also have similar outcomes. The strongest lesson is not that KNN should replace the earlier SVM or random-forest models. It is that **feature scaling, neighborhood size, and the definition of distance materially change what "similar customer" means**. For Milestone Two, I would keep the tuned KNN models as interpretable local-similarity benchmarks and compare them with the stronger nonlinear models from the earlier milestone.

## 15. Limitations

- KNN can become slow at prediction time because a new observation must be compared with many stored training observations.
- One-hot encoded categories increase dimensionality, and distance can become less informative as the number of dimensions grows.
- Results depend on how numeric and categorical information are represented in the same distance space.
- Cosine distance has a different interpretation from Euclidean/Manhattan distance and may not be ideal for every mixed-feature business problem.
- The validation-selected `k` values come from one development split; repeated cross-validation could be used for a more stable tuning estimate.
- The regression target `OrderCount` is discrete even though KNN regression returns a continuous average.
- The relationships are predictive, not causal.

## 16. Reproducibility notes

- Random seed: 42
- Data file: `data/churn_clean.csv`
- Classification target: `Churn`
- Regression target: `OrderCount`
- Train / validation / test split: 60% / 20% / 20%
- Classification split: stratified by `Churn`
- Numeric preprocessing: median imputation; standardized for the main distance comparisons
- Categorical preprocessing: most-frequent imputation + one-hot encoding
- Candidate `k`: 1, 3, 5, 7, 11, 15, 21, 31
- Distances: Euclidean, Manhattan, Minkowski (`p = 1.5`, `p = 3`), cosine
- Classification selection metric: validation ROC-AUC
- Regression selection metric: validation RMSE
- Test set used only after tuning choices were fixed